# Суммарный рейтинг по целевым группам в базе Big TV
Пример расчета суммарного рейтинга бренд-листов по нескольким целевым группам 

## Описание задачи и условий расчета
- Big TV Россия 0+
- Период: 01.02.2025 - 28.02.2025
- ЦА: Все 14-44, Все 45-55 BC, Все 18+, Все 25-34
- Место просмотра: Все места (Дом, Дача, Вне Дома)
- Каналы: все каналы
- Платформа: Все платформы (ТВ, Десктоп, Мобайл)
- Playback: Consolidated
- Ролики: Блок содержание - коммерческий; Блок распространение - орбитальный, сетевой; Статус события - реальный; Бренд - СБЕР; Ролик тип - ролик
- Статистики: SpotByBreaksRtg% (тотал), Quantity (тотал)

## Инициализация

При построении отчета первый шаг в любом ноутбуке - загрузка библиотек, которые помогут обращаться к TVI API и работать с данными.

Выполните следующую ячейку, для этого перейдите в нее и нажмите Ctrl+Enter

In [ ]:
%reload_ext autoreload
%autoreload 2

import sys
import os
import re
import json
import datetime
import time
import pandas as pd
#import matplotlib.pyplot as plt
from IPython.display import JSON
from mediascope_api.core import utils

from mediascope_api.core import net as mscore
from mediascope_api.mediavortex import tasks as cwt
from mediascope_api.mediavortex import catalogs as cwc

# Настраиваем отображение

# Включаем отображение всех колонок
pd.set_option('display.max_columns', None)
# Задаем максимальное количество выводимых строк. Раскомментируйте нужную строку
# 200 строк
# pd.set_option("display.max_rows", 200)
# Отображаем все строки. ВАЖНО! Отображение большого DataFrame требует много ресурсов
# pd.set_option("display.max_rows", None)

# Cоздаем объекты для работы с TVI API
mnet = mscore.MediascopeApiNetwork()
mtask = cwt.MediaVortexTask()
cats = cwc.MediaVortexCats()

## Справочники

Получим идентификаторы, которые будут использоваться для формирования условий расчета

In [ ]:
# Группа дохода по России
cats.get_tv_demo_attribute(names=['Группа дохода по России'])

In [ ]:
# Справочник мест просмотра
cats.get_tv_location()

In [ ]:
# Справочник платформ
cats.get_tv_platform()

In [ ]:
# Справочник Playback
cats.get_tv_playbacktype()

In [ ]:
# Блок содержание - коммерческий
cats.get_tv_breaks_content(name=['Коммерческий'])

In [ ]:
# Блок распространение - орбитальный, сетевой
cats.get_tv_breaks_distribution(name=['Орбитальный','Сетевой'])

In [ ]:
# Статус события - реальный
cats.get_tv_issue_status(name=['Реальный'])

In [ ]:
# Бренд - СБЕР
brand = cats.get_tv_brand(name=['СБЕР'])
brand[brand['name']=='СБЕР']

In [ ]:
# Ролик тип - ролик
cats.get_tv_ad_type(name=['ролик'])

## Формирование задания
Зададим условия расчета

In [ ]:
# Задаем период
# Период указывается в виде списка ('Начало', 'Конец') в формате 'YYYY-MM-DD'. Можно указать несколько периодов
date_filter = [('2025-02-01', '2025-02-28')]

# Задаем дни недели
weekday_filter = None

# Задаем тип дня
daytype_filter = None

# Задаем ЦА
basedemo_filter = None

# Доп фильтр ЦА, нужен только в случае расчета отношения между ЦА, например, при расчете Affinity Index
targetdemo_filter = None

# Задаем место просмотра
location_filter=None

# Задаем каналы
company_filter = None

# Указываем фильтр программ
program_filter = None

# Фильтр блоков
break_filter = 'breaksContentType = C and breaksDistributionType IN (O,N)'

# Фильтр роликов: статус - реальный, Товарная категория 4 уровня: КОЛБАСЫ И ВЕТЧИНЫ
ad_filter = 'adIssueStatusId = R and brandId = 1054587 and adTypeId = 1'

# Задаем платформу
platform_filter = None

# Задаем playback
playbacktype_filter = None

# Указываем список срезов (группировок)
slices = [
    'researchDate', # Дата (дни)
    'tvNetName', # Национальная телекомпания
    'brandListName', # Список брендов
]

# Указываем список статистик для расчета
statistics = ['SpotByBreaksRtgPerSum', 'QuantitySum']

# Задаем условия сортировки: Телекомпания (от а до я), рекламодатель (от а до я), бренд (от а до я), товар (от а до я)
sortings = {'researchDate':'ASC','tvNetName':'ASC','brandListName':'ASC'}

# Задаем опции расчета
options = {
    "kitId": 7, # Big TV Россия 0+
    "bigTv": True,
    "issueType": "AD", #Тип события - Ролики
}

Сформируем словарь с Playback

In [ ]:
playbacks = {
    'a. Consolidated' : 'playBackTypeId IN (0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28)',
}

Сформируем словарь с платформами

In [ ]:
platforms = {
    'a. TV, Desktop, Mobile': 'platformId IN (1,2,3)',
}

Сформируем словарь с местами просмотра

In [ ]:
locations = {
    'a. Дом, Дача, Вне Дома': 'locationId IN (1,2,4)',
}

Сформируем словарь с ЦА

In [ ]:
targets = {
    'a. Все 14-44': 'age >= 14 and age <= 44',
    'b. Все 45-55 BC': 'age >= 45 and age <= 55 and incomeGroupRussia IN (2,3)',
    'c. Все 18+': 'age >= 18',
    'd. Все 25-34': 'age >= 25 and age <= 34',
}

Создадим комбинации

In [ ]:
combinations = utils.combine_dicts(playbacks, platforms, locations, targets)

## Расчет задания

In [ ]:
# Посчитаем задания в цикле
tasks = []
print("Отправляем задания на расчет")

# Для каждой комбинации формируем задание и отправляем на расчет
for k, v in combinations.items():
    
    # Подставляем значения в параметры
    project_name = k
    
    playbacktype_filter = v[0] # Playback
    platform_filter = v[1] # платформа
    location_filter = v[2] # место просмотра
    basedemo_filter = v[3] # целевая аудитория
      
    # Формируем задание для API TV Index в формате JSON
    task_json = mtask.build_crosstab_task(date_filter=date_filter, weekday_filter=weekday_filter, 
                                          daytype_filter=daytype_filter, company_filter=company_filter, 
                                          location_filter=location_filter, basedemo_filter=basedemo_filter, 
                                          targetdemo_filter=targetdemo_filter,program_filter=program_filter, 
                                          break_filter=break_filter, ad_filter=ad_filter,
                                          platform_filter=platform_filter, playbacktype_filter=playbacktype_filter,
                                          slices=slices, statistics=statistics, sortings=sortings, options=options)

    # Для каждого этапа цикла формируем словарь с параметрами и отправленным заданием на расчет
    tsk = {}
    tsk['project_name'] = project_name    
    tsk['task'] = mtask.send_crosstab_task(task_json)
    tasks.append(tsk)
    time.sleep(2)
    print('.', end = '')
    
print(f"\nid: {[i['task']['taskId'] for i in tasks]}") 

print('')
# Ждем выполнения
print('Ждем выполнения')
tsks = mtask.wait_task(tasks)
print('Расчет завершен, получаем результат')

# Получаем результат
results = []
print('Собираем таблицу')
for t in tasks:
    tsk = t['task'] 
    df_result = mtask.result2table(mtask.get_result(tsk), project_name = t['project_name'])        
    results.append(df_result)
    print('.', end = '')
df = pd.concat(results)

# Приводим порядок столбцов в соответствие с условиями расчета
df = df[['prj_name']+slices+statistics]

df

In [ ]:
# Разобъем столбец с комбинациями
df[['playback','platform','location','target_audience']] = df['prj_name'].str.split(pat=';', n=3, expand=True)
df = df[['playback','platform','location','target_audience']+slices+statistics]
df

## Настройка внешнего вида таблицы
Пропустите этот шаг, если хотите экспортировать таблицу в ее текущем виде

In [ ]:
# Формируем сводную таблицу: строки - срезы; столбцы - место просмотра, Playback, платформа; значения - статистики
df = pd.pivot_table(df, values=statistics,
                        index=slices, 
                        columns=['playback','platform','location','target_audience'])
df

In [ ]:
# Опционально: перенести статистики в нижний уровень столбцов
df = df.reorder_levels([1, 2, 3, 4, 0], axis=1).sort_index(axis=1)

# Убираем объединение ячеек в строках 
df = df.reset_index()

df

## Сохраняем в Excel
По умолчанию файл сохраняется в папку `excel`

In [ ]:
df_info = mtask.task_builder.get_report_info()

with pd.ExcelWriter(mtask.task_builder.get_excel_filename('big_tv_crosstab_spots_02_rating_by_targets')) as writer:
    df.to_excel(writer, sheet_name='Report', index=True)
    df_info.to_excel(writer, sheet_name='Info', index=False)